# Exercise 01 - Pull real "claims" from the SEC `companyfacts` API

**Project Aegis - Phase 1 (Data & Storage) - learning loop step 2**

### Why this exercise
In Aegis, an *audit* is a `diff` between what a company **claims** (its filings) and the **evidence**
(its ledger). This notebook builds the **claims** side - the easy, real, public half - using the SEC's
structured **XBRL** data instead of parsing PDFs. By the end you'll have pulled JPMorgan's *real* total
liabilities as a clean number, which Exercise 02 reconciles a synthetic ledger against.

### What you'll learn
- **The claims layer** - what a public company asserts, and where it lives.
- **XBRL & the `companyfacts` API** - every reported number, machine-tagged, as JSON (no parsing).
- **Resolving a company to a CIK** - the SEC's primary key for a filer.
- **The anatomy of an XBRL fact** - `form`, `fy`, `fp`, `end`, `val`, `accn`, `frame`.
- The seed of the **`sec-xbrl` SourceConnector** (spec FR-IN-1, FR-IN-4): `resolve -> fetch -> normalize`.

### Requirements
- Internet access (we call live SEC endpoints).
- **Standard library only** - no `pip install` needed.

### How these exercises work
Code is explained in full, then you implement the marked gaps:
```python
# ==== YOUR CODE HERE (start) ====
...        # you write this
# ==== YOUR CODE HERE (end) ====
```
Each step ends with an `assert`-based self-check. Reference solutions live in
`learning/solutions/phase-1-data-storage/` - try first, then peek.

> **SEC etiquette (important).** The SEC requires a descriptive `User-Agent` header with a contact
> address on every request, and asks clients to stay under ~10 requests/second. Hammering the API
> without a proper UA gets your IP throttled. A real-world API-citizenship lesson, not a formality.

## Setup

Two tiny helpers: the required `User-Agent`, and a JSON-GET wrapper. Read them - you'll reuse `_get_json`.

In [1]:
import urllib.request, json

# The SEC wants a real contact in the UA. Edit the email to yours if you like.
SEC_USER_AGENT = {"User-Agent": "Aegis Learning Project sunitsingh.bitsg@gmail.com"}

def _get_json(url: str) -> dict:
    """GET a URL with the required SEC header and parse JSON."""
    req = urllib.request.Request(url, headers=SEC_USER_AGENT)
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.load(resp)

print("helpers ready")

helpers ready


## Step 1 - Resolve a ticker to a CIK

The SEC identifies every filer by a **CIK** (Central Index Key) - its primary key. Humans think in
tickers (`JPM`); the API thinks in CIKs (`0000019617`). The map is published at
`https://www.sec.gov/files/company_tickers.json`, whose values look like:

```json
{"cik_str": 19617, "ticker": "JPM", "title": "JPMORGAN CHASE & CO"}
```

Two gotchas you'll handle:
1. Compare tickers **case-insensitively**.
2. The `companyfacts` URL needs the CIK **zero-padded to 10 digits** (`19617` -> `0000019617`).

**Your task:** finish `resolve_cik` - find the matching record and return the padded CIK string.

In [10]:
def resolve_cik(ticker: str) -> str:
    """Return the 10-digit zero-padded CIK string for a ticker (e.g. 'JPM' -> '0000019617')."""
    table = _get_json("https://www.sec.gov/files/company_tickers.json")
    # `table` is a dict keyed by row index; each value has cik_str / ticker / title.
    # ==== YOUR CODE HERE (start) ====
    # 1) find the row whose 'ticker' matches `ticker` (case-insensitive)
    # 2) zero-pad its 'cik_str' to 10 digits and return it as a string
    table_values = list(table.values())
    for value in table_values:
        if value["ticker"].lower() == ticker.lower():
            return f"{value['cik_str']:010}"
    raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====

# --- self-check ---
cik = resolve_cik("JPM")
assert cik == "0000019617", f"expected 0000019617, got {cik!r}"
assert resolve_cik("jpm") == "0000019617", "should be case-insensitive"
print("PASS - JPM ->", cik)

PASS - JPM -> 0000019617


## Step 2 - Fetch the company's XBRL facts

`companyfacts` returns *every* number JPMorgan has ever reported, grouped as:

```
facts
  +- <taxonomy>            e.g. 'us-gaap', 'dei'
       +- <concept>        e.g. 'Liabilities', 'Assets', 'Revenues'
            +- label
            +- units
                 +- 'USD'  ->  [ {fact}, {fact}, ... ]   # one per reported period
```

This is the whole point of structured-first ingestion: **the claim is already a number** - no PDF, no
regex, no LLM. The function is given; run it and look at the shape.

In [20]:
def get_company_facts(cik: str) -> dict:
    """Fetch the full companyfacts document for a padded CIK."""
    return _get_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json")

facts = get_company_facts(cik)
print("entity     :", facts["entityName"])
print("taxonomies :", list(facts["facts"].keys()))
print("us-gaap concepts:", len(facts["facts"]["us-gaap"]))
print("Liabilities label:", facts["facts"]["us-gaap"]["Liabilities"]["label"])

entity     : JPMORGAN CHASE & CO
taxonomies : ['dei', 'invest', 'srt', 'us-gaap', 'ffd', 'ecd']
us-gaap concepts: 917
Liabilities label: Liabilities


## Step 3 - The anatomy of a fact, and extracting the latest annual claim

Each entry in `units['USD']` is one **reported datapoint**. The fields that matter:

| field | meaning |
|---|---|
| `val`  | the reported number (here, USD) |
| `end`  | the period-end date the value is *as of* (e.g. `2024-12-31`) |
| `form` | the filing it came from - `10-K` (annual), `10-Q` (quarterly) |
| `fy` / `fp` | fiscal year / fiscal period (`FY`, `Q1`...) |
| `accn` | accession number - the unique ID of the source filing (provenance!) |
| `frame` | an optional cross-company comparability tag |

A concept like `Liabilities` has **many** entries (restatements, amended filings, every year). For a
claims-vs-ledger check we want the **most recent annual (`10-K`) value**.

**Your task:** finish `latest_annual_value` - keep only `10-K` entries, pick the one with the latest
`end` date, and return a tidy record. (Hint: `max(seq, key=lambda r: r['end'])` works because ISO
dates sort lexicographically.)

In [21]:
def latest_annual_value(facts: dict, concept: str, taxonomy: str = "us-gaap") -> dict:
    """Return the most recent 10-K (annual) USD value for a concept, as a normalized record."""
    entries = facts["facts"][taxonomy][concept]["units"]["USD"]
    # ==== YOUR CODE HERE (start) ====
    # 1) keep only entries where form == '10-K'
    # 2) choose the entry with the latest 'end' date
    # 3) return a dict with keys: value, end, fy, form, accn
    latest_entry, latest_date = None, None
    for entry in entries:
        if entry["form"] == "10-K" and (not latest_date or entry['end'] > latest_date):
            latest_date = entry['end']
            latest_entry = entry
    return {
        "value": latest_entry["val"],   # rename val -> value, no mutation of the source
        "end":   latest_entry["end"],
        "fy":    latest_entry["fy"],
        "form":  latest_entry["form"],
        "accn":  latest_entry["accn"]
    }
    # raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====

# --- self-check (structural, so it survives new filings) ---
claim = latest_annual_value(facts, "Liabilities")
assert set(claim) >= {"value", "end", "fy", "form", "accn"}, "missing keys"
assert claim["form"] == "10-K", "must be the annual filing"
assert isinstance(claim["value"], int) and claim["value"] > 1_000_000_000_000, "JPM liabilities are > $1T"
print("PASS - latest annual Liabilities claim:")
print(f"  ${claim['value']:,}  as of {claim['end']}  (FY{claim['fy']}, {claim['form']}, accn {claim['accn']})")

PASS - latest annual Liabilities claim:
  $4,062,462,000,000  as of 2025-12-31  (FY2025, 10-K, accn 0001628280-26-008131)


## Step 4 - Normalize into a claim record (the connector's output)

Everything above is exactly what the **`sec-xbrl` SourceConnector** does in the real system
(spec FR-IN-1 / FR-IN-4): take a *descriptor* (company + concept), `resolve` the CIK, `fetch` the
facts, and `normalize` to a canonical record. Notice the record carries its own **provenance**
(`accn`, `end`) - that becomes the audit trail's grounding for the claim.

Run this to produce the structured claim that Exercise 02 reconciles against.

In [22]:
def pull_claim(ticker: str, concept: str) -> dict:
    """Mini SourceConnector: descriptor (ticker, concept) -> normalized claim record."""
    cik = resolve_cik(ticker)
    facts = get_company_facts(cik)
    rec = latest_annual_value(facts, concept)
    return {
        "source": "sec-xbrl",
        "entity": facts["entityName"],
        "cik": cik,
        "concept": concept,
        "claimed_value": rec["value"],
        "unit": "USD",
        "period_end": rec["end"],
        "provenance": {"form": rec["form"], "fy": rec["fy"], "accession": rec["accn"]},
    }

claim_record = pull_claim("JPM", "Liabilities")
print(json.dumps(claim_record, indent=2))

{
  "source": "sec-xbrl",
  "entity": "JPMORGAN CHASE & CO",
  "cik": "0000019617",
  "concept": "Liabilities",
  "claimed_value": 4062462000000,
  "unit": "USD",
  "period_end": "2025-12-31",
  "provenance": {
    "form": "10-K",
    "fy": 2025,
    "accession": "0001628280-26-008131"
  }
}


## Recap & what's next

You just built the **claims** half of an audit from real, structured, public data:
- resolved a ticker to its SEC **CIK**,
- pulled **XBRL** facts via `companyfacts` (no document parsing - *structured-first*, ADR-0001),
- extracted the **latest annual** value and normalized it with **provenance**.

This `claim_record` is the anchor for **Exercise 02**, where you'll generate a synthetic Faker ledger
whose transactions sum to `claimed_value` (a clean book) or to a deliberately different total (a
**seeded discrepancy**) - and write the reconciliation that flags the delta. That's SG-1, end to end.

**Stretch goals (optional):**
1. Generalize `latest_annual_value` to accept `10-Q` and return the latest quarter.
2. Pull `Assets` and `StockholdersEquity` too, and check `Assets ~= Liabilities + Equity` (the
   balance-sheet identity) on the latest 10-K.
3. Add a 0.2s sleep between requests to be a good API citizen under batch pulls.